In [ ]:
# ─────────────────────────────────────────────
# BLOCK 1: Setup & Configuration
# Mount drive, install libraries, define keywords
# ─────────────────────────────────────────────

# Install pytesseract and tesseract-ocr
!pip install pytesseract Pillow==9.0.0
!sudo apt update
!sudo apt install tesseract-ocr

from google.colab import drive
drive.mount('/content/drive')

import os
import re
import csv
import requests
import pandas as pd
import xml.etree.ElementTree as ET
from PIL import Image, ImageEnhance
import pytesseract

# ── Paths ──────────────────────────────────────────────────
INPUT_CSV     = '/content/drive/MyDrive/final_ugp/scopus.csv'          # your input file (read-only)
XML_DIR       = '/content/drive/MyDrive/final_ugp/pipeline/xmls'       # downloaded XML files
CSV_DIR       = '/content/drive/MyDrive/final_ugp/pipeline/csvs'       # per-paper figure CSVs
RELEVANT_OUT  = '/content/drive/MyDrive/final_ugp/pipeline/relevant_papers.csv'
TRACKER_OUT   = '/content/drive/MyDrive/final_ugp/pipeline/tracker.csv'

for d in [XML_DIR, CSV_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Elsevier API key ───────────────────────────────────────
API_KEY = 'Your_Els_API_Key_Here'  # Replace with your actual API key

# ── Keywords to search for ─────────────────────────────────
KEYWORDS = [
    'capacity', 'voltage', 'current', 'charge', 'discharge',
    'cycle life', 'impedance', 'specific capacity', 'plot', 'curve'
]

print("✓ Setup complete. Paths and keywords are ready.")

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:9 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,295 kB]
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,945 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,061 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,016 kB]
G

In [2]:
# ─────────────────────────────────────────────
# BLOCK 2: Install Tesseract OCR
# Only needs to run once per Colab session
# ─────────────────────────────────────────────

!apt-get update -qq
!apt-get install -y -qq tesseract-ocr libtesseract-dev
!tesseract --version

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libarchive-dev:amd64.
(Reading database ... 118252 files and directories currently installed.)
Preparing to unpack .../libarchive-dev_3.6.0-1ubuntu1.6_amd64.deb ...
Unpacking libarchive-dev:amd64 (3.6.0-1ubuntu1.6) ...
Selecting previously unselected package libleptonica-dev.
Preparing to unpack .../libleptonica-dev_1.82.0-3build1_amd64.deb ...
Unpacking libleptonica-dev (1.82.0-3build1) ...
Selecting previously unselected package libtesseract-dev:amd64.
Preparing to unpack .../libtesseract-dev_4.1.1-2.1build1_amd64.deb ...
Unpacking libtesseract-dev:amd64 (4.1.1-2.1build1) ...
Setting up libleptonica-dev (1.82.0-3build1) ...
Setting up libarchive-dev:amd64 (3.6.0-1ubuntu1.6) ...
Setting up libtesseract-dev:amd64 (4.1.1-2.1build1) ...
Processing triggers for ma

In [3]:
# ─────────────────────────────────────────────
# BLOCK 3: Helper Functions
# Reusable utilities used in Stages 1 and 2
# ─────────────────────────────────────────────

def keyword_scan(text, keywords):
    """
    Checks if any keyword appears as a whole word in the given text.
    Returns (found: bool, matched_keywords: list).
    """
    if not isinstance(text, str):
        return False, []
    found_keywords = []
    for kw in keywords:
        if re.search(r'\b' + re.escape(kw) + r'\b', text, re.IGNORECASE):
            found_keywords.append(kw)
    return bool(found_keywords), found_keywords


def fetch_xml(doi, api_key):
    """
    Fetches the full-text XML for a paper from the Elsevier API.
    Returns (status_code, response_content_or_None).
    """
    url = f'https://api.elsevier.com/content/article/doi/{doi}?view=FULL'
    headers = {'X-ELS-APIKEY': api_key, 'Accept': 'text/xml'}
    try:
        r = requests.get(url, stream=True, headers=headers, timeout=30)
        return r.status_code, r if r.status_code == 200 else None
    except Exception as e:
        print(f"  [fetch_xml] Error for DOI {doi}: {e}")
        return 0, None


def parse_figures_from_xml(xml_path):
    """
    Parses an Elsevier XML file and extracts figure metadata:
    attachment EID, UCS locator, filename, and caption text.
    Returns a list of dicts.
    """
    tree = ET.parse(xml_path)
    root = tree.getroot()

    NS_CE   = {'ce':   'http://www.elsevier.com/xml/common/dtd'}
    NS_XOCS = {'xocs': 'http://www.elsevier.com/xml/xocs/dtd'}

    # --- Captions ------------------------------------------------
    captions = []
    for figure in root.findall('.//ce:figure', NS_CE):
        cap_el = figure.find('.//ce:caption', NS_CE)
        if cap_el is not None:
            sp_el = cap_el.find('.//ce:simple-para', NS_CE)
            if sp_el is not None:
                raw = ET.tostring(sp_el, encoding='unicode', method='text')
                captions.append(" ".join(re.sub('<.*?>', '', raw).split()))
            else:
                captions.append(None)
        else:
            captions.append(None)

    # --- Attachments (images) ------------------------------------
    attachments = []
    for att in root.findall('.//xocs:attachment', NS_XOCS):
        try:
            eid      = att.find('.//xocs:attachment-eid', NS_XOCS).text
            locator  = att.find('.//xocs:ucs-locator',    NS_XOCS).text
            filename = att.find('.//xocs:filename',        NS_XOCS).text
            attachments.append({'eid': eid, 'locator': locator, 'filename': filename})
        except AttributeError:
            continue  # skip malformed entries

    # --- Zip captions and attachments together -------------------
    figures = []
    for i, att in enumerate(attachments):
        caption = captions[i] if i < len(captions) else None
        link    = f"https://ars.els-cdn.com/content/image/{att['eid']}".replace('.jpg', '_lrg.jpg')
        figures.append({
            'UTD_EID':     att['eid'],
            'UCS_Locator': att['locator'],
            'Filename':    att['filename'],
            'Caption':     caption,
            'link':        link,
        })
    return figures


def ocr_image_from_url(url, keywords):
    """
    Downloads an image directly into memory, runs Tesseract OCR,
    and scans the extracted text for keywords.
    Returns (status_code, extracted_text, matched_keywords).
      status_code: 2 = keywords found, 0 = no keywords, 1 = error
    """
    try:
        response = requests.get(url, stream=True, timeout=30)
        if response.status_code != 200:
            return 1, '', []

        # Load image from bytes without saving to disk
        from io import BytesIO
        image = Image.open(BytesIO(response.content)).convert('L')
        image = ImageEnhance.Contrast(image).enhance(2)
        text  = pytesseract.image_to_string(image)

        found, matched = keyword_scan(text, keywords)
        return (2 if found else 0), text, matched

    except Exception as e:
        print(f"  [ocr_image_from_url] Error: {e}")
        return 1, '', []

print("✓ Helper functions defined.")

✓ Helper functions defined.


In [4]:
# ─────────────────────────────────────────────
# BLOCK 4: Stage 1 — Caption Keyword Filter
# Downloads XML for each paper, parses figure
# captions, and checks for keywords.
# Does NOT save images.
# ─────────────────────────────────────────────

df_input = pd.read_csv(INPUT_CSV)

# We work on a fresh copy — original scopus.csv is never touched
df = df_input.copy()

stage1_results = []  # per-paper results for the tracker

for i, row in df.iterrows():
    doi = row.get('DOI', '')
    print(f"[{i+1}/{len(df)}] DOI: {doi}")

    if pd.isna(doi) or str(doi).strip() == '':
        stage1_results.append({'index': i, 'stage1_status': 'skipped', 'stage1_matched_kws': ''})
        continue

    # ── Fetch XML ──────────────────────────────────────────────
    status_code, response = fetch_xml(str(doi).strip(), API_KEY)

    if status_code != 200:
        print(f"  → HTTP {status_code}: XML not available")
        stage1_results.append({'index': i, 'stage1_status': 'no_xml', 'stage1_matched_kws': ''})
        continue

    # ── Save XML to disk (needed for Stage 2 parsing) ──────────
    xml_path = os.path.join(XML_DIR, f'{i+1}.xml')
    with open(xml_path, 'wb') as f:
        for chunk in response.iter_content(2048):
            f.write(chunk)

    # ── Parse figures & scan captions ─────────────────────────
    try:
        figures = parse_figures_from_xml(xml_path)
    except ET.ParseError as e:
        print(f"  → ParseError: {e}")
        stage1_results.append({'index': i, 'stage1_status': 'parse_error', 'stage1_matched_kws': ''})
        continue

    # Save per-paper figure CSV (used in Stage 2)
    figure_csv_path = os.path.join(CSV_DIR, f'output{i+1}.csv')
    pd.DataFrame(figures).to_csv(figure_csv_path, index=False)

    # Check if any caption contains a keyword
    all_matched = []
    for fig in figures:
        found, matched = keyword_scan(fig.get('Caption', ''), KEYWORDS)
        fig['stage1_hit']      = 1 if found else 0
        fig['stage1_keywords'] = ', '.join(matched)
        all_matched.extend(matched)

    # Save updated figure CSV with stage1 columns
    pd.DataFrame(figures).to_csv(figure_csv_path, index=False)

    paper_hit = bool(all_matched)
    status    = 'passed_stage1' if paper_hit else 'dropped_stage1'
    print(f"  → Stage 1: {'PASS' if paper_hit else 'DROP'} | Keywords: {list(set(all_matched))}")
    stage1_results.append({
        'index':               i,
        'stage1_status':       status,
        'stage1_matched_kws':  ', '.join(set(all_matched)),
    })

# ── Attach Stage 1 results to main dataframe ──────────────
s1_df = pd.DataFrame(stage1_results).set_index('index')
df = df.join(s1_df)

print(f"\n✓ Stage 1 complete.")
print(f"  Passed : {(df['stage1_status'] == 'passed_stage1').sum()}")
print(f"  Dropped: {(df['stage1_status'] == 'dropped_stage1').sum()}")

Streaming output truncated to the last 5000 lines.
[2503/5000] DOI: 10.1039/d5ra02951a
  → HTTP 404: XML not available
[2504/5000] DOI: 10.1038/s41467-024-51196-x
  → HTTP 404: XML not available
[2505/5000] DOI: 10.1016/j.apenergy.2024.123671
  → HTTP 400: XML not available
[2506/5000] DOI: 10.1038/s41598-017-13740-2
  → HTTP 404: XML not available
[2507/5000] DOI: 10.1039/d1ma00572c
  → HTTP 404: XML not available
[2508/5000] DOI: 10.1039/d2nr02807g
  → HTTP 404: XML not available
[2509/5000] DOI: 10.1016/j.jcis.2023.04.005
  → HTTP 400: XML not available
[2510/5000] DOI: 10.1002/cphc.202100700
  → HTTP 404: XML not available
[2511/5000] DOI: 10.3390/met14030345
  → HTTP 404: XML not available
[2512/5000] DOI: 10.1007/s10008-023-05514-w
  → HTTP 404: XML not available
[2513/5000] DOI: 10.1109/ACCESS.2018.2825466
  → HTTP 404: XML not available
[2514/5000] DOI: 10.1016/j.egypro.2017.09.508
  → Stage 1: DROP | Keywords: []
[2515/5000] DOI: 10.1149/1945-7111/ad1d98
  → HTTP 404: XML not 

In [5]:
# ─────────────────────────────────────────────
# BLOCK 5: Stage 2 — OCR Keyword Filter
# For papers that passed Stage 1, downloads
# images TEMPORARILY (in memory only),
# runs Tesseract OCR, and scans for keywords.
# No images are saved to disk.
# ─────────────────────────────────────────────

stage2_results = []

# Only process papers that passed Stage 1
stage1_passed = df[df['stage1_status'] == 'passed_stage1']
print(f"Running Stage 2 on {len(stage1_passed)} papers...\n")

for i, row in stage1_passed.iterrows():
    figure_csv_path = os.path.join(CSV_DIR, f'output{i+1}.csv')

    if not os.path.exists(figure_csv_path):
        print(f"[{i+1}] Figure CSV not found — skipping")
        stage2_results.append({'index': i, 'stage2_status': 'no_figure_csv', 'stage2_matched_kws': ''})
        continue

    dof = pd.read_csv(figure_csv_path)
    print(f"[{i+1}] Processing {len(dof[dof['stage1_hit'] == 1])} stage-1 figures...")

    paper_matched = []

    for j, fig_row in dof[dof['stage1_hit'] == 1].iterrows():
        url = fig_row['link']
        ocr_status, text, matched = ocr_image_from_url(url, KEYWORDS)

        dof.at[j, 'stage2_ocr_status']   = ocr_status
        dof.at[j, 'stage2_ocr_text']      = text
        dof.at[j, 'stage2_keywords']      = ', '.join(matched)
        dof.at[j, 'stage2_hit']           = 1 if ocr_status == 2 else 0

        if ocr_status == 2:
            print(f"  Figure {j}: OCR MATCH — {matched}")
            paper_matched.extend(matched)

    # Save updated figure CSV with stage2 columns
    dof.to_csv(figure_csv_path, index=False)

    paper_hit = bool(paper_matched)
    status    = 'passed_stage2' if paper_hit else 'dropped_stage2'
    print(f"  → Stage 2: {'PASS' if paper_hit else 'DROP'}\n")

    stage2_results.append({
        'index':               i,
        'stage2_status':       status,
        'stage2_matched_kws':  ', '.join(set(paper_matched)),
    })

# ── Attach Stage 2 results to main dataframe ──────────────
s2_df = pd.DataFrame(stage2_results).set_index('index')
df = df.join(s2_df)

# Papers that never reached Stage 2 get a clear label
df['stage2_status']      = df['stage2_status'].fillna('not_processed_stage2')
df['stage2_matched_kws'] = df['stage2_matched_kws'].fillna('')

print(f"\n✓ Stage 2 complete.")
print(f"  Passed : {(df['stage2_status'] == 'passed_stage2').sum()}")
print(f"  Dropped: {(df['stage2_status'] == 'dropped_stage2').sum()}")

Running Stage 2 on 566 papers...

[11] Processing 2 stage-1 figures...
  → Stage 2: DROP

[19] Processing 2 stage-1 figures...
  → Stage 2: DROP

[25] Processing 1 stage-1 figures...
  → Stage 2: DROP

[30] Processing 3 stage-1 figures...
  → Stage 2: DROP

[31] Processing 1 stage-1 figures...
  → Stage 2: DROP

[34] Processing 1 stage-1 figures...
  → Stage 2: DROP

[36] Processing 9 stage-1 figures...
  → Stage 2: DROP

[43] Processing 7 stage-1 figures...
  Figure 6: OCR MATCH — ['capacity', 'current', 'cycle life', 'impedance']
  Figure 10: OCR MATCH — ['discharge']
  → Stage 2: PASS

[117] Processing 3 stage-1 figures...
  → Stage 2: DROP

[127] Processing 2 stage-1 figures...
  → Stage 2: DROP

[139] Processing 2 stage-1 figures...
  → Stage 2: DROP

[145] Processing 5 stage-1 figures...
  Figure 1: OCR MATCH — ['impedance']
  → Stage 2: PASS

[155] Processing 4 stage-1 figures...
  Figure 5: OCR MATCH — ['capacity']
  Figure 6: OCR MATCH — ['charge', 'discharge']
  Figure 7: OCR

In [6]:
# ─────────────────────────────────────────────
# BLOCK 6: Save Outputs
# 1. tracker.csv  — every paper with status at
#    each stage and reason for dropping
# 2. relevant_papers.csv — papers that passed
#    both stages, with full original metadata
# ─────────────────────────────────────────────

# ── Derive a single human-readable "final_status" column ──
def get_final_status(row):
    s1 = row.get('stage1_status', '')
    s2 = row.get('stage2_status', '')
    if s1 in ('no_xml', 'parse_error', 'skipped'):
        return f'dropped — {s1}'
    if s1 == 'dropped_stage1':
        return 'dropped — no caption keywords (stage 1)'
    if s2 == 'dropped_stage2':
        return 'dropped — no OCR keywords (stage 2)'
    if s2 == 'passed_stage2':
        return 'relevant'
    return 'undetermined'

df['final_status'] = df.apply(get_final_status, axis=1)

# ── tracker.csv — full picture of every paper ─────────────
tracker_cols = [
    'Authors', 'Author full names', "Author(s) ID",
    'Year', 'Cited by', 'DOI', 'Link',
    'stage1_status', 'stage1_matched_kws',
    'stage2_status', 'stage2_matched_kws',
    'final_status',
]
# Keep only columns that actually exist in df
tracker_cols = [c for c in tracker_cols if c in df.columns]
df[tracker_cols].to_csv(TRACKER_OUT, index=False)
print(f"✓ Tracker saved → {TRACKER_OUT}")

# ── relevant_papers.csv — only the papers that made it ────
relevant = df[df['final_status'] == 'relevant'].copy()
relevant.to_csv(RELEVANT_OUT, index=False)
print(f"✓ Relevant papers saved → {RELEVANT_OUT}")
print(f"\n  Total papers    : {len(df)}")
print(f"  Relevant        : {len(relevant)}")
print(f"  Dropped stage 1 : {(df['stage1_status'] == 'dropped_stage1').sum()}")
print(f"  Dropped stage 2 : {(df['stage2_status'] == 'dropped_stage2').sum()}")
print(f"  No XML / errors : {df['stage1_status'].isin(['no_xml','parse_error','skipped']).sum()}")

✓ Tracker saved → /content/drive/MyDrive/final_ugp/pipeline/tracker.csv
✓ Relevant papers saved → /content/drive/MyDrive/final_ugp/pipeline/relevant_papers.csv

  Total papers    : 5000
  Relevant        : 314
  Dropped stage 1 : 305
  Dropped stage 2 : 252
  No XML / errors : 4129
